# Data Project - part 2.3 + 2.4 + 2.5

Questions 2.1 and 2.2 are contained in the notebook of part 2.1 + 2.2, which builds
on the module `model_2_1.py`.

Questions 2.4 and 2.5 additionally use `model_2_4.py`. This is a flexible version of
the same model, written as a function, so that the individual mechanisms can be
switched off one at a time and an extra source of risk can be added. The module
reproduces the baseline simulation in `model_2_1.py` exactly when it is called with
its default arguments, and it draws the same random numbers in the same order in all
simulations. Differences across simulations are therefore caused by the mechanism
that is changed, and not by different random draws.

**Imports:** We now import all the modules, we need for this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# autoreload modules when code is run
%load_ext autoreload
%autoreload 2

# user written modules
import model_2_1
import model_2_4

## Question 2.3 - Compute the Gini coefficient

We write our own function for the Gini coefficient. The function sorts the incomes in
ascending order and uses the formula

$$ G = \frac{\sum_{i=1}^{n}(2i - n - 1)\,y_{(i)}}{n \sum_{i=1}^{n} y_{(i)}} $$

where $y_{(i)}$ is the $i$'th smallest income. We also write a function that returns
the Lorenz curve.

In [ ]:
# 2.3: functions for the Gini coefficient and the Lorenz curve

def gini(y):
    """ compute the Gini coefficient of a vector of incomes

    Args:
        y (ndarray): vector of incomes (non-negative)

    Returns:
        (float): the Gini coefficient
    """

    # flatten and sort the incomes in ascending order
    y = np.sort(np.asarray(y, dtype=float).ravel())

    # number of observations
    n = y.size

    # ranks from 1 to n
    i = np.arange(1, n + 1)

    # covariance-based formula for the Gini coefficient
    return np.sum((2 * i - n - 1) * y) / (n * np.sum(y))


def lorenz(y):
    """ compute the Lorenz curve of a vector of incomes

    Args:
        y (ndarray): vector of incomes (non-negative)

    Returns:
        F (ndarray): cumulative population share
        L (ndarray): cumulative income share
    """

    # flatten and sort the incomes in ascending order
    y = np.sort(np.asarray(y, dtype=float).ravel())

    # cumulative income share, starting from the origin
    L = np.concatenate(([0.0], np.cumsum(y) / np.sum(y)))

    # cumulative population share, starting from the origin
    F = np.linspace(0, 1, y.size + 1)

    return F, L

We test the function on three cases where the answer is known analytically: a uniform
distribution on $[0,1]$, where the Gini coefficient is exactly $1/3$; a lognormal
distribution whose logarithm has standard deviation $s$, where it is
$2\Phi(s/\sqrt{2}) - 1$; and perfect equality, where it is zero.

In [ ]:
# 2.3: test the Gini function on cases where the answer is known

rng_test = np.random.default_rng(1917)

# a. uniform distribution on [0,1]
y_uniform = rng_test.uniform(0, 1, size=1_000_000)

print("uniform   - simulated:", round(gini(y_uniform), 4),
      "theoretical:", round(1/3, 4))

# b. lognormal distribution
s = 0.5
y_lognormal = rng_test.lognormal(0, s, size=1_000_000)

print("lognormal - simulated:", round(gini(y_lognormal), 4),
      "theoretical:", round(2 * norm.cdf(s / np.sqrt(2)) - 1, 4))

# c. perfect equality
print("equality  - simulated:", gini(np.ones(1000)),
      "theoretical:", 0.0)

The function returns the correct answer in all three cases, so we can use it on the
simulated income distribution.

2.3.1

In [ ]:
# 2.3.1: Gini coefficient and Lorenz curve for the full simulated sample

# pool all individuals and all ages
income_pooled = model_2_1.income.ravel()

# compute the Gini coefficient for the pooled sample
gini_pooled = gini(income_pooled)

print("Gini coefficient, all ages pooled:", round(gini_pooled, 4))

# compute the Lorenz curve
F, L = lorenz(income_pooled)

# plot the Lorenz curve together with the line of perfect equality
plt.plot(F, L, label='Lorenz curve')
plt.plot([0, 1], [0, 1], linestyle='--', color='black', label='Perfect equality')

# add labels and title
plt.xlabel('Cumulative population share')
plt.ylabel('Cumulative income share')
plt.title('Lorenz curve, all ages pooled')

# add legend
plt.legend()

# show the figure
plt.show()

2.3.2

In [ ]:
# 2.3.2: Gini coefficient within each age group

# create an empty list to store the Gini coefficient for each age
gini_by_age = []

# compute the Gini coefficient separately for each age
for t, age in enumerate(model_2_1.ages):
    gini_by_age.append(gini(model_2_1.income[:, t]))

# convert to an array for convenience
gini_by_age = np.array(gini_by_age)

# plot the within-age Gini coefficient over the life cycle
plt.plot(model_2_1.ages, gini_by_age, label='Within age group')

# add the pooled Gini coefficient as a horizontal reference line
plt.axhline(gini_pooled, linestyle='--', color='black', label='All ages pooled')

# add labels and title
plt.xlabel('Age')
plt.ylabel('Gini coefficient')
plt.title('Inequality over the life cycle')

# add legend
plt.legend()

# show the figure
plt.show()

# print the Gini coefficient at selected ages
for age in [18, 25, 35, 45, 60, 65]:
    print(f"Age {age}:", round(gini_by_age[age - 18], 4))

# find the age where the within-age measure crosses the pooled measure
crossing_age = model_2_1.ages[np.argmax(gini_by_age > gini_pooled)]

print("Crossing age:", crossing_age)

**Analysis:**
The Gini coefficient for the pooled sample is 0.378, and the Lorenz curve lies clearly
below the 45-degree line.

Within a given age group, inequality follows a pronounced life-cycle profile. It is
zero at age 18, where everybody is still in education and receives the same student
grant, and then rises almost monotonically from 0.196 at age 25 to 0.452 at age 65, as
the multiplicative shocks accumulate and unemployment spells depreciate human capital.

The comparison with the pooled measure is therefore not one-directional. Up to age 51
the within-age Gini coefficient is *below* the pooled one, because pooling adds a
between-age component: a 60-year-old employed worker and a 20-year-old student differ
in income for purely life-cycle reasons, and this variation is counted as inequality
when all ages are pooled. From age 52 onwards the within-age coefficient *exceeds* the
pooled one, because by then the dispersion accumulated within the cohort is larger than
the dispersion generated by pooling ages together.

The within-age measure is the more meaningful notion of inequality here, since it
compares individuals who are otherwise similar. We return to this distinction in
question 2.4.

## Question 2.4 - What drives inequality?

We now run a series of alternative simulations where the components of the model are
switched off one at a time. We first check that `model_2_4.simulate()` reproduces the
baseline exactly.

In [ ]:
# 2.4: check that the flexible module reproduces the baseline exactly

baseline = model_2_4.simulate()

print("identical income:", np.array_equal(baseline['income'], model_2_1.income))
print("identical employment:", np.array_equal(baseline['employed'], model_2_1.employed))

In [ ]:
# 2.4: switch off the components of the model one at a time

specifications = {
    'Baseline': {},
    'No educational differences': {'educational_differences': False},
    'No shocks to human capital': {'shocks': False},
    'No depreciation when unemployed': {'depreciation': False},
    'No unemployment': {'unemployment': False},
}

# select a single age group for the within-age comparison
age_group = 45
t_group = age_group - 18

# create an empty dictionary to store the results
results = {}

# run the alternative simulations
for name, switches in specifications.items():

    sim = model_2_4.simulate(**switches)

    results[name] = {
        'pooled': gini(sim['income']),
        'within': gini(sim['income'][:, t_group]),
        'mean': sim['income'].mean(),
    }

# print the results as a table
print(f"{'Simulation':34s}{'Pooled':>9s}{'Age 45':>9s}{'Mean y':>9s}")

for name, r in results.items():
    print(f"{name:34s}{r['pooled']:9.4f}{r['within']:9.4f}{r['mean']:9.3f}")

In [ ]:
# 2.4: plot the change in the Gini coefficient relative to the baseline

# names of the alternative simulations
names = [name for name in specifications if name != 'Baseline']

# changes relative to the baseline
d_pooled = [results[n]['pooled'] - results['Baseline']['pooled'] for n in names]
d_within = [results[n]['within'] - results['Baseline']['within'] for n in names]

# position of the bars
x = np.arange(len(names))
width = 0.35

# plot the two sets of bars next to each other
plt.bar(x - width/2, d_pooled, width, label='All ages pooled')
plt.bar(x + width/2, d_within, width, label=f'Age {age_group}')

# add a horizontal line at zero
plt.axhline(0, color='black', linewidth=0.8)

# add labels and title
plt.xticks(x, names, rotation=30, ha='right')
plt.ylabel('Change in Gini coefficient')
plt.title('Change in inequality relative to the baseline')

# add legend
plt.legend()

# adjust spacing
plt.tight_layout()

# show the figure
plt.show()

**Analysis:**
The mechanism that matters by far the most is the idiosyncratic shock to human capital.
Switching it off lowers the pooled Gini coefficient from 0.378 to 0.279 and the
within-age Gini coefficient at age 45 from 0.340 to 0.214, so roughly a third of the
inequality at that age disappears. The reason is that $\psi$ enters multiplicatively
and is never reversed: each shock permanently shifts the level of the individual's
human capital, so the variance of log human capital grows with the number of years
worked. Educational differences come second, reducing the pooled Gini coefficient by
0.045 and the within-age one by 0.048. Depreciation and unemployment have small effects.

The last two simulations illustrate exactly the case mentioned in the hint. Switching
off depreciation, or unemployment altogether, *lowers* the within-age Gini coefficient
at age 45 (from 0.340 to 0.329 and 0.326 respectively) but *raises* the pooled Gini
coefficient (from 0.378 to 0.383 and 0.382). Within an age group the direction is the
intuitive one: unemployment and the associated loss of human capital are a source of
dispersion between otherwise identical workers, so removing them compresses the
distribution.

The pooled measure moves in the opposite direction because it also contains a
between-age component. Removing unemployment means that individuals work every year
after entry and accumulate human capital without interruption, which raises average
income from 1.47 to 1.82. The income of individuals still in education is fixed at the
student grant of 0.45 and does not move at all. The gap between young students and
older workers therefore widens, and this larger between-age dispersion more than offsets
the smaller within-age dispersion. This is a good illustration of why the pooled Gini
coefficient is a poor measure of inequality in a life-cycle model: it mixes genuine
inequality between individuals with mechanical differences between age groups.

One caveat: switching off educational differences requires a choice of which education
everybody receives. We assign the medium education. The pooled Gini coefficient is
sensitive to this choice (0.298 with short, 0.333 with medium and 0.382 with long
education, since the length of education determines how many years are spent on the
fixed student grant), but the within-age coefficient at age 45 is stable between 0.284
and 0.300. The conclusion is therefore unaffected.

## Question 2.5 - Extension: more risk

We extend the model with **permanent health risk**. In each year after labor market
entry, an individual is hit by a health shock with probability $p^{h}$. The shock is
absorbing: the individual never works again and receives the replacement rate of the
income in the last job, exactly as an unemployed person does, but without any
possibility of returning to employment.

The interesting feature of this extension is that the risk is *permanent*, whereas
unemployment in the baseline model is *temporary*. The health shock is drawn from a
separate random number generator, so that $p^{h} = 0$ reproduces the baseline exactly.

In [ ]:
# 2.5: simulate the model for different degrees of health risk

# values of the annual probability of a health shock
p_health_values = [0.000, 0.005, 0.010, 0.020, 0.030]

# create empty lists to store the results
ever_disabled = []
mean_income_health = []
gini_pooled_health = []
gini_age_health = []

# index of age 60
t60 = 60 - 18

# run one simulation for each value of p_health
for p in p_health_values:

    sim = model_2_4.simulate(p_health=p)

    # share of individuals who are disabled at the end of the life cycle
    ever_disabled.append(sim['disabled'][:, -1].mean())

    mean_income_health.append(sim['income'].mean())
    gini_pooled_health.append(gini(sim['income']))
    gini_age_health.append(gini(sim['income'][:, t60]))

# print the results as a table
print(f"{'p_health':>10s}{'Disabled':>10s}{'Mean y':>10s}{'Pooled':>10s}{'Age 60':>10s}")

for i, p in enumerate(p_health_values):
    print(f"{p:10.3f}{ever_disabled[i]:10.3f}{mean_income_health[i]:10.3f}"
          f"{gini_pooled_health[i]:10.4f}{gini_age_health[i]:10.4f}")

In [ ]:
# 2.5: Gini coefficient over the life cycle with and without health risk

# baseline and extended simulation
sim_base = model_2_4.simulate()
sim_health = model_2_4.simulate(p_health=0.010)

# compute the within-age Gini coefficient in both simulations
gini_age_base = np.array([gini(sim_base['income'][:, t])
                          for t in range(len(model_2_4.ages))])
gini_age_ext = np.array([gini(sim_health['income'][:, t])
                         for t in range(len(model_2_4.ages))])

# plot the two life-cycle profiles
plt.plot(model_2_4.ages, gini_age_base, label='Baseline')
plt.plot(model_2_4.ages, gini_age_ext, label='With health risk')

# add labels and title
plt.xlabel('Age')
plt.ylabel('Gini coefficient')
plt.title('Within-age inequality with and without health risk')

# add legend
plt.legend()

# show the figure
plt.show()

In [ ]:
# 2.5: compare the income distribution at age 60 with and without health risk

# use the same bins for both histograms
bins = np.linspace(0, np.percentile(sim_base['income'][:, t60], 99), 60)

# plot the two distributions on top of each other
plt.hist(sim_base['income'][:, t60], bins=bins, alpha=0.6, label='Baseline')
plt.hist(sim_health['income'][:, t60], bins=bins, alpha=0.6, label='With health risk')

# add labels and title
plt.xlabel('Income')
plt.ylabel('Number of individuals')
plt.title('Income distribution at age 60')

# add legend
plt.legend()

# show the figure
plt.show()

# compare the disabled with the rest of the population at age 60
is_disabled = sim_health['disabled'][:, t60]
income_60 = sim_health['income'][:, t60]

print("Share disabled at age 60:", round(is_disabled.mean(), 3))
print("Mean income, disabled:", round(income_60[is_disabled].mean(), 3))
print("Mean income, others:", round(income_60[~is_disabled].mean(), 3))

In [ ]:
# 2.5: is it the amount of non-employment or its persistence that matters?

# non-employment rate at age 60 in the simulation with health risk
in_labor_market = model_2_4.ages[t60] >= sim_health['entry_age']

target = (
    (in_labor_market & ~sim_health['employed'][:, t60]).sum()
    / in_labor_market.sum()
)

# a purely temporary risk with the same steady-state non-employment rate
# requires a job-separation probability of sigma = u*lambda/(1-u)
sigma_alt = target * model_2_4.job_finding / (1 - target)

sim_temporary = model_2_4.simulate(job_separation_alt=sigma_alt)

print("Non-employment rate at age 60:", round(target, 4))
print("Implied job-separation probability:", round(sigma_alt, 4))
print()

print(f"{'':22s}{'Pooled':>9s}{'Age 60':>9s}{'Mean y':>9s}")

print(f"{'Permanent risk':22s}{gini(sim_health['income']):9.4f}"
      f"{gini(sim_health['income'][:, t60]):9.4f}{sim_health['income'].mean():9.3f}")

print(f"{'Temporary risk':22s}{gini(sim_temporary['income']):9.4f}"
      f"{gini(sim_temporary['income'][:, t60]):9.4f}{sim_temporary['income'].mean():9.3f}")

**Analysis:**
Health risk raises inequality within age groups throughout the life cycle. With
$p^{h} = 0.01$, 36.5 per cent of the cohort has been hit by a health shock by age 65,
and the Gini coefficient at age 60 rises from 0.428 to 0.447. The effect grows with
age, since the probability of ever having been hit is cumulative: at age 25 the Gini
coefficient rises only from 0.196 to 0.211, and at age 45 from 0.340 to 0.361. The
histogram at age 60 shows why. The distribution becomes bimodal, with a mass of
disabled individuals whose income is frozen at 60 per cent of the wage they earned when
the shock arrived, while everybody else continues to accumulate human capital. At age
60 the disabled earn 0.88 on average against 2.00 for the rest of the population.

The pooled Gini coefficient behaves non-monotonically: it rises slightly for small
$p^{h}$ but falls below the baseline for $p^{h} \geq 0.02$. This is the same mechanism
as in question 2.4, running in reverse. Health shocks reduce average income, from 1.47
to 1.08 at $p^{h} = 0.03$, and thereby push the incomes of older workers down towards
the fixed student grant of 0.45. This compresses the between-age component of the
pooled measure even while inequality within each age group keeps rising. It confirms
that the pooled Gini coefficient should not be read as a measure of inequality between
individuals in this model.

Finally, the last comparison shows that the persistence of the risk matters in itself,
and not only the amount of non-employment it generates. Setting $\sigma$ so that the
non-employment rate at age 60 is the same as under health risk gives a pooled Gini
coefficient of 0.350 against 0.380, so concentrating the same amount of non-employment
permanently on a minority generates more measured inequality than spreading it
temporarily across everybody. The within-age difference at age 60 is much smaller
(0.447 against 0.434), so the effect works mainly through the lower tail and the shape
of the distribution rather than through the within-age Gini coefficient. The comparison
also has a clear limitation: the temporary version destroys far more human capital in
the aggregate (average income 0.80 against 1.30), because with a separation probability
of 0.37 essentially everybody spends long spells depreciating.